# 03 — MLflow: training и tracking

На предыдущем этапе были сравнены разные стратегии разбиения данных и несколько моделей.

В этом ноутбуке переносим **финальный экспериментальный протокол в MLflow**.

Основной validation protocol:

- `StratifiedGroupKFold`;
- 4 fold;
- группы = последовательные экспериментальные блоки;
- `cycle_id` не используется как признак.

Сравниваются три модели:

- Logistic Regression — baseline;
- Random Forest;
- HistGradientBoosting.

Для каждого из четырёх компонентов запускается отдельный MLflow Run.

Главная метрика — `f1_macro_overall`, рассчитанная по всем out-of-fold предсказаниям.


## 1. Импорты и настройки

MLflow Tracking Server уже запущен в Docker и доступен локально по адресу `http://localhost:5001`.

Все последующие runs будут отправляться в experiment `hydraulic-classification`.


In [ ]:
import hashlib
import json
import time
from pathlib import Path

import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from sklearn.ensemble import (
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

MLFLOW_TRACKING_URI = "http://localhost:5001"
EXPERIMENT_NAME = "hydraulic-classifier"

RANDOM_STATE = 42
N_SPLITS = 4

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

print(f"MLflow URI: {MLFLOW_TRACKING_URI}")
print(f"Experiment: {EXPERIMENT_NAME}")

## 2. Загрузка подготовленных данных

Используем те же `features` и `profile`, которые были получены на предыдущих этапах проекта.

Важно сохранить ту же самую версию признаков, что использовалась в `02_classifier.ipynb`, чтобы результаты MLflow были сопоставимы с проведённым исследованием.


In [ ]:
DATA_DIR = Path("../data/processed")

FEATURES_PATH = DATA_DIR / "X.csv"
PROFILE_PATH = DATA_DIR / "y.csv"

features = pd.read_csv(FEATURES_PATH)
profile = pd.read_csv(PROFILE_PATH)

print("features:", features.shape)
print("profile: ", profile.shape)

display(features.head())
display(profile.head())

### Проверка данных

На вход модели должны поступать только признаки сенсоров.

`cycle_id` сохраняется для анализа порядка эксперимента, но исключается из `X`, поскольку он связан с временной структурой датасета.


In [ ]:
X = features.drop(columns=["cycle_id"])

targets = [
    "cooler",
    "valve",
    "pump",
    "accumulator",
]

print(f"X shape: {X.shape}")
print(f"Targets: {targets}")

## 3. Формирование экспериментальных блоков

Последовательные циклы объединяются в один group, пока не изменится хотя бы один из параметров.

Это предотвращает попадание частей одного экспериментального режима одновременно в train и test.

Полученные группы используются в `StratifiedGroupKFold`.


In [ ]:
regime = profile[["cooler", "valve", "pump", "accumulator"]]

groups = regime.ne(regime.shift()).any(axis=1).cumsum()

print(f"Number of experimental blocks: {groups.nunique()}")
print("Block sizes:")
print(groups.value_counts().value_counts().sort_index())

## 4. Cross-validation protocol

Для каждого target используем один и тот же принцип:

- 4 fold;
- stratification по текущему target;
- grouping по экспериментальным блокам;
- в каждом fold модель обучается только на train blocks;
- текущий fold используется для получения out-of-fold predictions.


In [ ]:
cv = StratifiedGroupKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

print(f"CV: StratifiedGroupKFold(n_splits={N_SPLITS})")

## 5. Модели

Используем три алгоритма.

**Logistic Regression** — baseline и линейная модель.

**Random Forest** — ансамбль независимых решающих деревьев.

**HistGradientBoosting** — градиентный бустинг, выбранный как основной кандидат по результатам предыдущего сравнения.

Для Logistic Regression стандартизация выполняется внутри `Pipeline`, чтобы scaler обучался только на train fold. Деревья масштабирования не требуют.


In [ ]:
def make_models():  # noqa: ANN201
    return {
        "logistic_regression": Pipeline(
            [
                ("scaler", StandardScaler()),
                (
                    "classifier",
                    LogisticRegression(
                        max_iter=2000,
                        random_state=RANDOM_STATE,
                    ),
                ),
            ]
        ),
        "random_forest": RandomForestClassifier(
            n_estimators=300,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        "hist_gradient_boosting": HistGradientBoostingClassifier(
            max_iter=200,
            learning_rate=0.1,
            max_leaf_nodes=31,
            random_state=RANDOM_STATE,
        ),
    }


models = make_models()
list(models)

## 6. Dataset tracking

Для воспроизводимости фиксируем:

- источник датасета;
- версию;
- количество объектов и признаков;
- список target'ов;
- pipeline подготовки признаков;
- SHA-256 digest подготовленного датасета.

Digest позволяет проверить, что в разных запусках использовалась одна и та же версия данных.


In [ ]:
def dataframe_sha256(df):  # noqa: ANN001, ANN201
    data = pd.util.hash_pandas_object(
        df,
        index=True,
    ).values.tobytes()
    return hashlib.sha256(data).hexdigest()


dataset_for_tracking = pd.concat(
    [features, profile],
    axis=1,
)

dataset_digest = dataframe_sha256(dataset_for_tracking)

dataset_metadata = {
    "source": "UCI Condition Monitoring of Hydraulic Systems",
    "dataset_version": "UCI Dataset 447",
    "n_samples": int(dataset_for_tracking.shape[0]),
    "n_features": int(features.shape[1]),
    "n_targets": int(profile.shape[1]),
    "feature_pipeline": [
        "data_loader.py",
        "features.py",
    ],
    "targets": targets,
    "dataset_digest": dataset_digest,
}

print(json.dumps(dataset_metadata, indent=2))

## 7. MLflow experiment

Каждая комбинация `target × model` будет отдельным Run.

В каждом Run сохраняем:

### Parameters
- target;
- model;
- число fold'ов;
- число признаков;
- random state;
- параметры модели.

### Metrics
- `f1_macro_overall` — основная метрика;
- `f1_weighted_overall`;
- mean/std по fold'ам для accuracy и F1;
- training time.

### Artifacts
- classification report;
- confusion matrix;
- fold-level metrics;
- dataset metadata.



In [ ]:
RESULTS_DIR = Path("mlflow_results")
RESULTS_DIR.mkdir(exist_ok=True)


def log_model_params(model):  # noqa: ANN001, ANN201
    params = {}

    if isinstance(model, Pipeline):
        classifier = model.named_steps["classifier"]
        params["classifier"] = classifier.__class__.__name__
        params.update(
            {
                f"classifier.{key}": value
                for key, value in classifier.get_params().items()
                if isinstance(value, (str, int, float, bool, type(None)))
            }
        )
    else:
        params["classifier"] = model.__class__.__name__
        params.update(
            {
                f"classifier.{key}": value
                for key, value in model.get_params().items()
                if isinstance(value, (str, int, float, bool, type(None)))
            }
        )

    return params

## 8. Запуск всех MLflow Runs

Для каждого target:

1. создаём новый экземпляр модели;
2. выполняем 4-fold grouped cross-validation;
3. сохраняем prediction каждого fold;
4. считаем fold-level metrics;
5. объединяем все OOF predictions;
6. считаем одну `f1_macro_overall` на всех 2205 объектах;
7. логируем результаты в MLflow.

Таким образом, каждый Run полностью воспроизводит соответствующий эксперимент из `02_classifier.ipynb`.


In [ ]:
all_results = []

for target in targets:
    y = profile[target]

    for model_name in models:
        model = make_models()[model_name]

        fold_results = []
        oof_predictions = np.empty(len(y), dtype=y.dtype)

        start_time = time.perf_counter()

        for fold, (train_idx, test_idx) in enumerate(
            cv.split(X, y, groups=groups),
            start=1,
        ):
            X_train = X.iloc[train_idx]
            X_test = X.iloc[test_idx]
            y_train = y.iloc[train_idx]
            y_test = y.iloc[test_idx]

            fold_model = make_models()[model_name]

            fold_start = time.perf_counter()
            fold_model.fit(X_train, y_train)
            y_pred = fold_model.predict(X_test)
            fold_time = time.perf_counter() - fold_start

            oof_predictions[test_idx] = y_pred

            fold_results.append(
                {
                    "fold": fold,
                    "accuracy": accuracy_score(y_test, y_pred),
                    "f1_macro": f1_score(
                        y_test,
                        y_pred,
                        average="macro",
                    ),
                    "f1_weighted": f1_score(
                        y_test,
                        y_pred,
                        average="weighted",
                    ),
                    "training_time_sec": fold_time,
                }
            )

        total_time = time.perf_counter() - start_time

        fold_df = pd.DataFrame(fold_results)

        overall_f1_macro = f1_score(
            y,
            oof_predictions,
            average="macro",
        )

        overall_f1_weighted = f1_score(
            y,
            oof_predictions,
            average="weighted",
        )

        overall_accuracy = accuracy_score(
            y,
            oof_predictions,
        )

        with mlflow.start_run(run_name=f"{target}__{model_name}"):
            mlflow.set_tags(
                {
                    "target": target,
                    "model": model_name,
                    "split": "experimental_blocks",
                    "validation": "StratifiedGroupKFold",
                    "dataset.source": dataset_metadata["source"],
                    "dataset.version": dataset_metadata["dataset_version"],
                    "dataset.digest": dataset_digest,
                    "feature.pipeline": "data_loader.py + features.py",
                }
            )

            mlflow.log_params(
                {
                    "target": target,
                    "model": model_name,
                    "split": "experimental_blocks",
                    "cv": "StratifiedGroupKFold",
                    "n_splits": N_SPLITS,
                    "n_samples": len(X),
                    "n_features": X.shape[1],
                    "n_groups": groups.nunique(),
                    "random_state": RANDOM_STATE,
                    **log_model_params(model),
                }
            )

            mlflow.log_metrics(
                {
                    "accuracy_overall": overall_accuracy,
                    "f1_macro_overall": overall_f1_macro,
                    "f1_weighted_overall": overall_f1_weighted,
                    "accuracy_mean": fold_df["accuracy"].mean(),
                    "accuracy_std": fold_df["accuracy"].std(ddof=0),
                    "f1_macro_mean": fold_df["f1_macro"].mean(),
                    "f1_macro_std": fold_df["f1_macro"].std(ddof=0),
                    "f1_weighted_mean": fold_df["f1_weighted"].mean(),
                    "f1_weighted_std": fold_df["f1_weighted"].std(ddof=0),
                    "training_time_sec": total_time,
                }
            )

            run_dir = RESULTS_DIR / target / model_name
            run_dir.mkdir(parents=True, exist_ok=True)

            fold_path = run_dir / "fold_metrics.csv"
            fold_df.to_csv(fold_path, index=False)

            report = classification_report(
                y,
                oof_predictions,
                output_dict=True,
            )

            report_path = run_dir / "classification_report.json"
            report_path.write_text(
                json.dumps(report, indent=2),
                encoding="utf-8",
            )

            cm = confusion_matrix(y, oof_predictions)
            cm_path = run_dir / "confusion_matrix.csv"
            pd.DataFrame(cm).to_csv(
                cm_path,
                index=False,
            )

            mlflow.log_artifacts(
                str(run_dir),
                artifact_path="evaluation",
            )
            mlflow.log_dict(
                dataset_metadata,
                "dataset/dataset_metadata.json",
            )

            # После CV обучаем финальную модель
            # на всех доступных данных.
            final_model = make_models()[model_name]
            final_model.fit(X, y)

            mlflow.sklearn.log_model(
                final_model,
                name="model",
                skops_trusted_types=[
                    "sklearn.tree._tree.Tree",
                    "sklearn.ensemble._hist_gradient_boosting.predictor.TreePredictor",
                ],
            )

            all_results.append(
                {
                    "target": target,
                    "model": model_name,
                    "run_id": mlflow.active_run().info.run_id,
                    "accuracy_overall": overall_accuracy,
                    "f1_macro_overall": overall_f1_macro,
                    "f1_weighted_overall": overall_f1_weighted,
                    "f1_macro_mean": fold_df["f1_macro"].mean(),
                    "f1_macro_std": fold_df["f1_macro"].std(ddof=0),
                    "training_time_sec": total_time,
                }
            )

## 9. Сводная таблица

Здесь сравниваем все 12 Runs.

`f1_macro_overall` является основной метрикой, поскольку macro F1 одинаково учитывает все классы и не позволяет большим классам полностью определять итоговую оценку.


In [ ]:
results_df = pd.DataFrame(all_results)

results_df = results_df.sort_values(
    ["target", "f1_macro_overall"],
    ascending=[True, False],
)

display(results_df.round(4))

summary_path = RESULTS_DIR / "mlflow_runs_summary.csv"
results_df.to_csv(summary_path, index=False)

print(f"Saved: {summary_path}")

## 10. Лучший кандидат для каждого target

Выбираем модель с максимальным `f1_macro_overall` отдельно для каждого компонента.


In [ ]:
best_models = (
    results_df.sort_values(
        ["target", "f1_macro_overall"],
        ascending=[True, False],
    )
    .groupby("target", as_index=False)
    .first()
)

display(
    best_models[
        [
            "target",
            "model",
            "f1_macro_overall",
            "f1_weighted_overall",
            "f1_macro_mean",
            "f1_macro_std",
        ]
    ].round(4)
)

### Предварительный итог

В MLflow зарегистрированы результаты всех комбинаций:

Для каждого Run сохранены:

- параметры эксперимента;
- dataset digest и lineage;
- fold-level metrics;
- overall OOF metrics;
- classification report;
- confusion matrix;
- обученная модель.

На следующем этапе выбранную модель можно перенести в **Model Registry**, присвоить alias `champion` и использовать в inference-приложении.


In [ ]:
import mlflow
from mlflow import MlflowClient

client = MlflowClient(tracking_uri="http://localhost:5001")

MODEL_NAMES = {
    "cooler": "hydraulic_cooler",
    "valve": "hydraulic_valve",
    "pump": "hydraulic_pump",
    "accumulator": "hydraulic_accumulator",
}

for _, row in best_models.iterrows():
    name = MODEL_NAMES[row["target"]]
    version = mlflow.register_model(
        f"runs:/{row['run_id']}/model",
        name,
    )
    client.set_registered_model_alias(
        name,
        "champion",
        version.version,
    )
    print(f"{name}: v{version.version} → champion")

### Доп (1): проверка inference

In [ ]:
sample = features.drop(columns=["cycle_id"]).iloc[0].to_dict()

print("Количество признаков:", len(sample))
print(sample)

In [ ]:
from IPython.display import JSON, display

display(JSON({"features": sample}))

In [ ]:
import requests

response = requests.post(
    "http://localhost:8000/api/v1/process",
    json={"features": sample},
)

print("Status:", response.status_code)
print("Response:", response.json())

In [ ]:
true_values = profile.iloc[0][["cooler", "valve", "pump", "accumulator"]].to_dict()

print("True:", true_values)
print("Pred:", response.json()["predictions"])

In [ ]:
bad_sample = {"features": {"PS1_mean": sample["PS1_mean"]}}

response = requests.post(
    "http://localhost:8000/api/v1/process",
    json=bad_sample,
)

print(response.status_code)
print(response.json())

In [ ]:
bad_sample = sample.copy()
bad_sample["some_fake_feature"] = 123.0

response = requests.post(
    "http://localhost:8000/api/v1/process",
    json={"features": bad_sample},
)

print(response.status_code)
print(response.json())

### Доп: замена champion

In [ ]:
experiment = client.get_experiment_by_name("hydraulic-classifier")

runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="tags.target = 'pump'",
    order_by=["metrics.f1_macro_overall DESC"],
)

for run in runs:
    print(
        f"run_id={run.info.run_id} | "
        f"model={run.data.tags.get('model')} | "
        f"f1_macro={run.data.metrics.get('f1_macro_overall')}"
    )

In [ ]:
best_run = runs[0]
second_run = runs[1]

print("Champion Run:")
print(best_run.info.run_id, best_run.data.tags.get("model"))

print("\nSecond Run:")
print(second_run.info.run_id, second_run.data.tags.get("model"))

In [ ]:
second_run_id = second_run.info.run_id

new_version = mlflow.register_model(
    model_uri=f"runs:/{second_run_id}/model",
    name="hydraulic_pump",
)

print(f"Создана hydraulic_pump v{new_version.version}")

In [ ]:
client.set_registered_model_alias(
    "hydraulic_pump",
    "champion",
    "1",
)

In [ ]:
champion = client.get_model_version_by_alias(
    "hydraulic_pump",
    "champion",
)

print("Champion version:", champion.version)
print("Champion run:", champion.run_id)